META DATASET

In [1]:
from google.colab import drive
import os

# Montar el drive
drive.mount('/content/drive')

Mounted at /content/drive


META DATASET JSON

In [8]:
import os
import json
from osgeo import gdal
import glob

# 1. Búsqueda automática del archivo
ruta_busqueda = '/content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/E3/'
# Buscamos cualquier archivo que contenga "Dataset_Maestro" y termine en .tif
archivos_encontrados = glob.glob(os.path.join(ruta_busqueda, "**/*Dataset_Maestro*.tif"), recursive=True)

if not archivos_encontrados:
    print("❌ No se encontró ningún archivo que coincida. Por favor, verifica que el archivo se haya guardado correctamente en Drive.")
else:
    master_path = archivos_encontrados[0]
    json_path = os.path.join(os.path.dirname(master_path), 'Metadata_Dataset.json')
    print(f"🔍 Archivo localizado en: {master_path}")

    # 2. Procesar Metadatos
    ds = gdal.Open(master_path)
    if ds:
        metadata = {
            "project": "Modelo de predicción de inundaciones en Veracruz",
            "spatial_reference": ds.GetProjection(),
            "dimensions": {"width": ds.RasterXSize, "height": ds.RasterYSize, "res": 1.5},
            "bands_config": []
        }

        band_info = [
            {"id": 1, "name": "Elevación (DTM)", "unit": "msnm", "type": "Continuous"},
            {"id": 2, "name": "Edafología", "unit": "Category_ID", "type": "Categorical"},
            {"id": 3, "name": "Infraestructura (Vialidades)", "unit": "Binary", "type": "Boolean"},
            {"id": 4, "name": "Capa Urbana (Exposición)", "unit": "Binary", "type": "Boolean"}
        ]

        for i, info in enumerate(band_info, start=1):
            band = ds.GetRasterBand(i)
            stats = band.GetStatistics(True, True)
            metadata["bands_config"].append({
                "band_index": i,
                "name": info["name"],
                "statistics": {"min": round(stats[0], 2), "max": round(stats[1], 2), "mean": round(stats[2], 2)}
            })

        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(metadata, f, indent=4, ensure_ascii=False)

        print(f"✅ Metadata_Dataset.json generado con éxito en la misma carpeta.")
    else:
        print("❌ Error al abrir el dataset con GDAL.")

🔍 Archivo localizado en: /content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/E3/Dataset Maestro /Dataset_Maestro_Multicanal_Veracruz.tif
✅ Metadata_Dataset.json generado con éxito en la misma carpeta.
